# 📘 Notebook 15: Sequence Models & the Road to Attention

### Course: *From Python to Deep Learning & Fine-Tuning*
**Instructor:** Ioannis Tsioulis

*Module D: Deep Learning · Notebook 4 of 4 in this module · (15 of 18 overall)*

---

## 🎯 Learning objectives

By the end of this notebook you will be able to:

- Understand why **sequential data** (text, time series) needs special handling
- Explain the idea of a **recurrent neural network (RNN)** and its memory
- Identify the **limitations** of RNNs that motivated something better
- Grasp the core intuition of **attention**: focusing on the relevant parts of the input
- See why attention leads directly to the **Transformer** (next module)

> **Prerequisites:** Notebooks 12-14.
>
> 🔭 **Why this notebook is a turning point.** Everything so far assumed inputs of fixed size with no order. But language and time series are *sequences*: order matters and length varies. Solving this well led to **attention**, the idea behind the Transformer, and therefore behind the large language models that power tools like the one you are reading this in.

> ℹ️ **Setup note.** Mostly conceptual; code cells use NumPy/PyTorch: `import piplite; await piplite.install(['torch','numpy','matplotlib'])`

## 1. What makes sequential data different

### Order carries meaning
Consider two sentences with the same words: *‘the dog bit the man’* and *‘the man bit the dog’*. Identical words, opposite meaning, the **order** is everything. A CNN or a plain feed-forward network has no natural notion of sequence order or of variable length. Sequential data needs a model that processes inputs **in order** and carries information forward.

### Examples of sequences
Text (a sequence of words), audio (a sequence of sound samples), stock prices or weather (time series), DNA (a sequence of bases). All share the property that earlier elements provide context for later ones.

## 2. Recurrent Neural Networks (RNNs)

### Intuition first
An **RNN** reads a sequence one element at a time and maintains a **hidden state**, a memory summarising what it has seen so far. At each step it combines the current input with its previous hidden state to produce a new hidden state. In effect, the network has a loop: it feeds its own memory back into itself.

### The update, in words and symbols
At step $t$, with input $x_t$ and previous memory $h_{t-1}$:

$$h_t = f(W_x x_t + W_h h_{t-1} + b)$$

The same weights $W_x, W_h$ are reused at every step (parameter sharing again, like CNNs across space, here across time). The hidden state $h_t$ is the network's running summary of the sequence.

In [ ]:
import numpy as np

np.random.seed(42)
# A tiny RNN cell processing a sequence of 4 numbers, by hand
Wx = np.random.randn(3, 1) * 0.5   # input -> hidden (3 hidden units)
Wh = np.random.randn(3, 3) * 0.5   # hidden -> hidden
b  = np.zeros(3)

sequence = [1.0, 2.0, 3.0, 4.0]
h = np.zeros(3)                     # initial memory is empty

for t, x in enumerate(sequence):
    h = np.tanh(Wx.flatten() * x + Wh @ h + b)   # update memory
    print(f"After input {x}: hidden state = {h.round(3)}")

Notice how the hidden state evolves as each new input arrives, it accumulates context. The **same** weights process every time step. More capable variants, **LSTM** and **GRU**, add gates that help the network remember information over longer spans, but the core loop is the same.

## 3. Where RNNs fall short

RNNs were the standard for sequences for years, but they have two stubborn problems:

1. **Long-range memory fades.** Information from early in a long sequence must survive many update steps to influence a late output. In practice it tends to get diluted, the network 'forgets' distant context (the *vanishing gradient* problem, related to the gradient issues from Notebook 12).

2. **They are inherently sequential.** Because step $t$ depends on step $t-1$, the computation **cannot be parallelised** across the sequence. On modern hardware built for parallelism, this makes RNNs slow to train on long sequences.

These two limitations set the stage for a fundamentally different idea.

## 4. The core idea of attention

### Intuition first
When you translate the sentence *‘the cat sat on the mat’* into another language and you are producing the word for *‘cat’*, you focus your **attention** on the word *‘cat’* in the source, not equally on every word. **Attention** gives a network the same ability: for each output, it learns *which parts of the input to focus on*, and how much.

### Why this is powerful
Attention lets the model connect any two positions **directly**, no matter how far apart, solving the long-range memory problem at a stroke. And because every position can attend to every other **at once**, the computation **parallelises**, solving the speed problem too. It addresses *both* RNN weaknesses simultaneously.

### A first look at attention weights
At its heart, attention computes a set of **weights** (one per input position) that sum to 1, then forms a weighted average of the inputs. The weights say 'how much to focus here'. Let us compute a simple version:

In [ ]:
import numpy as np

def softmax(v):
    e = np.exp(v - np.max(v))    # subtract max for numerical stability
    return e / e.sum()

# A "query" (what we are looking for) and several "keys" (what each input offers)
query = np.array([1.0, 0.0])
keys = np.array([[1.0, 0.0],    # very similar to the query
                 [0.0, 1.0],    # unrelated
                 [0.9, 0.1]])   # fairly similar

# Attention scores = how well each key matches the query (a dot product, again!)
scores = keys @ query
weights = softmax(scores)

print("Raw scores:", scores.round(3))
print("Attention weights (sum to 1):", weights.round(3))

The input most similar to the query received the largest weight, the model 'pays attention' to it. The matching is just a **dot product** (Notebook 05 once more), and `softmax` turns the scores into weights that sum to 1. This small mechanism, scaled up and repeated, is the engine of the Transformer.

> 🧠 **The thread that runs through everything.** Notice that attention, like neurons, convolutions, and RNN updates, is built from the **dot product**. From Notebook 05 onward, this single operation, measuring alignment between vectors, keeps reappearing. Deep learning is remarkably economical in its fundamental ideas.

---
## ✏️ Exercises

### Exercise 1
Run the hand-coded RNN cell with a longer sequence, e.g. `[1, 2, 3, 4, 5, 6, 7, 8]`. Observe how the hidden state keeps changing. In one sentence, explain why information from the very first input might be hard to recover by the last step.

In [ ]:
# Modify the sequence and re-run; then write your explanation as a comment.


<details><summary>💡 Show solution</summary>

```python
# Each step overwrites the hidden state via a tanh of a weighted mix. After many
# steps, the contribution of the first input has been transformed repeatedly and
# largely diluted -- this is the long-range memory / vanishing-gradient problem.
```

*This is precisely the weakness attention removes by connecting positions directly.*
</details>

### Exercise 2
Using the `softmax` function above, compute attention weights for `query = [0, 1]` against the same `keys`. Which input now receives the most attention, and does that match your intuition?

In [ ]:
import numpy as np
# (Re-use softmax and keys.)
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
query = np.array([0.0, 1.0])
scores = keys @ query
weights = softmax(scores)
print(weights.round(3))   # the second key [0, 1] now dominates
```

*Changing the query changes the focus. This query-dependent focusing is exactly what makes attention flexible.*
</details>

## 🔑 Key takeaways

- **Sequential data** depends on order and varies in length, needing models that process inputs in sequence.
- An **RNN** carries a **hidden state** (memory), updated at each step with shared weights.
- RNNs struggle with **long-range memory** and **cannot be parallelised** across the sequence.
- **Attention** lets a model focus on the most relevant input positions, connecting distant positions directly and in parallel.
- Attention is built from the familiar **dot product** plus **softmax**, the foundation of the Transformer.

### 🏁 You have completed Module D: Deep Learning!
From a single neuron to CNNs and the threshold of attention, you now understand how modern networks are built and trained. The final module brings it all together into the architecture behind today's most capable AI.

**Next:** *Module E: Transformers & Fine-Tuning*, starting with the Transformer architecture itself (Notebook 16).

---
*End of Module D.*

---
*© Ioannis Tsioulis. *From Python to Deep Learning & Fine-Tuning*. Prepared for educational use.*